In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))
    
    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble import ensemble
from src.model.game_model_collection import GamingModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metric = ClassificationMetrics()

In [2]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


# Fit the Gaming Text Preprocessor 

In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

In [4]:
# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

In [5]:
# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [6]:
# train 
gaming_preprocessor.fit(train, text_column=tc)

In [7]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

val = pd.concat([wot_val, dota_val], ignore_index=True)

# Label Cleaning

In [8]:

# convert dota labels 
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
        # data_source_column='data_source',
    output_column='label_binary'
)
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
        # data_source_column='data_source',
    output_column='label_binary'
)


In [9]:
train["label_binary"].value_counts()

label_binary
0    40929
1    12778
Name: count, dtype: int64

# Model Loading

In [10]:
# load all models from disk
model_dir = CONFIG['model_dir'] / "binary"
model_paths = list(model_dir.glob("gaming_all_*.joblib"))
models = [joblib.load(path) for path in model_paths]

model_paths

[PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/gaming_all_binary_models.joblib')]

# Simple Ensemble

In [11]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = model_paths,
)

In [12]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False)'])

In [13]:
# instantiate the ensemble with model paths
binary_ensemble = ensemble.Ensemble(
    model_collections=[gaming_model_collection]
)

In [14]:
# run simple majority voting on the training data
pred_train_binary_ensemble = binary_ensemble.predict_simple_majority(train[tc])
metric.metrics(train['label_binary'], pred_train_binary_ensemble)

Predicting with SimpleMajority...
Input has 53707 samples.
wot_Logistic_Regression: (53707,)
wot_LinearSVC: (53707,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (53707,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (53707,)
dota_Logistic_Regression: (53707,)
dota_LinearSVC: (53707,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (53707,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (53707,)
combined_Logistic_Regression: (53707,)
combined_LinearSVC: (53707,)
combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (53707,)
combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False): (53707,)
Collected predictions from 12 models.
Prediction matrix shape: (53707, 12)
Aggregating predictions from 12 models...


{'CV Macro F1': 0.9680192940424788,
 'CV Weighted F1': 0.976793580913903,
 'Accuracy': 0.9767814251401121,
 'Coverage': np.float64(1.0),
 'Precision': 0.9498322540376063,
 'FPR': np.float64(0.015710132180116788),
 'FNR': np.float64(0.04726874315229301),
 'TPR': np.float64(0.952731256847707),
 'TNR': np.float64(0.9842898678198833)}

In [15]:
# run simple majority voting on the validation data
pred_val_binary_ensemble = binary_ensemble.predict_simple_majority(val[tc])
metric.metrics(val['label_binary'], pred_val_binary_ensemble)

Predicting with SimpleMajority...
Input has 17056 samples.
wot_Logistic_Regression: (17056,)
wot_LinearSVC: (17056,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (17056,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (17056,)
dota_Logistic_Regression: (17056,)
dota_LinearSVC: (17056,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (17056,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (17056,)
combined_Logistic_Regression: (17056,)
combined_LinearSVC: (17056,)
combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True): (17056,)
combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False): (17056,)
Collected predictions from 12 models.
Prediction matrix shape: (17056, 12)
Aggregating predictions from 12 models...


{'CV Macro F1': 0.8424220814419543,
 'CV Weighted F1': 0.8938676621627305,
 'Accuracy': 0.8949343339587242,
 'Coverage': np.float64(1.0),
 'Precision': 0.7728958630527818,
 'FPR': np.float64(0.05962100217212194),
 'FNR': np.float64(0.2688259109311741),
 'TPR': np.float64(0.7311740890688259),
 'TNR': np.float64(0.940378997827878)}

# Fitted Ensemble

In [16]:
thresholds = np.arange(0.50, 0.99, 0.05)

In [17]:
res = binary_ensemble.fit_weighted_confidence_majority(X_val=train[tc], y_val=train['label_binary'], score_func=metric.score, thresholds=thresholds)


In [18]:
weights, threshold, _ = res

train_pred = binary_ensemble.predict_weighted_confidence_majority(train[tc], weights=weights, threshold=threshold)
metric.metrics(train['label_binary'], train_pred)

{'CV Macro F1': 0.9801735306682493,
 'CV Weighted F1': 0.9882234858412862,
 'Accuracy': 0.9883334053493497,
 'Coverage': np.float64(0.8618243431954866),
 'Precision': 0.9913537549407114,
 'FPR': np.float64(0.0018523418893887271),
 'FNR': np.float64(0.05532015065913371),
 'TPR': np.float64(0.9446798493408662),
 'TNR': np.float64(0.9981476581106112)}

In [19]:
val_pred = binary_ensemble.predict_weighted_confidence_majority(val[tc], weights=weights, threshold=threshold)
metric.metrics(val['label_binary'], val_pred)

{'CV Macro F1': 0.8814468197715586,
 'CV Weighted F1': 0.9293681299496219,
 'Accuracy': 0.9315868060268766,
 'Coverage': np.float64(0.8638602251407129),
 'Precision': 0.8764805414551607,
 'FPR': np.float64(0.024443328310731625),
 'FNR': np.float64(0.2568149210903874),
 'TPR': np.float64(0.7431850789096126),
 'TNR': np.float64(0.9755566716892684)}

In [20]:
weights, threshold

(array([0.04523348, 0.00946077, 0.08420375, 0.03204597, 0.09418352,
        0.03581342, 0.00528644, 0.04582971, 0.49255951, 0.02053279,
        0.05121582, 0.08363481]),
 np.float64(0.8000000000000003))